# Post-Correction Pipeline – Konfidenz + Qwen-Korrektur

**Stufe 1 — Konfidenz-Extraktion (TrOCR)**  
Jedes transkribierte Wort erhält einen Konfidenzwert (geometrisches Mittel der Token-Log-Probs).  
Ausgabe: `data/confidence/<page_id>.txt`

**Stufe 2 — Postkorrektur (Qwen)**  
Wörter unter dem Schwellwert werden mit `<<wort>>` markiert.  
Qwen korrigiert im Kontext: das Ersatzwort soll inhaltlich passen und dem Original optisch ähneln.  
Ausgabe: `data/corrected_transcriptions/<page_id>.txt`

**Stufe 3 — Gesamtdokument**  
Ausgabe: `data/corrected_raw_document.txt`

> Beide Stufen sind **resumable** – bereits vorhandene Dateien werden übersprungen.

In [1]:
import gc
import json
import re
import warnings
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm

warnings.filterwarnings('ignore')

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

# --- Pfade ---
TROCR_CKPT      = REPO_ROOT / 'checkpoints' / 'best_cook_large_v4'
MANIFEST_PATH   = REPO_ROOT / 'data' / 'all_manifests' / 'manifest.json'
TRANSCR_DIR     = REPO_ROOT / 'data' / 'transcriptions'
CONF_DIR        = REPO_ROOT / 'data' / 'confidence'
CORRECTED_DIR   = REPO_ROOT / 'data' / 'corrected_transcriptions'
DEBUG_DIR       = REPO_ROOT / 'data' / 'debug_logs'          # Step-1-Vorschläge als JSON
CORR_DOC_PATH   = REPO_ROOT / 'data' / 'corrected_raw_document.txt'

CONF_DIR.mkdir(parents=True, exist_ok=True)
CORRECTED_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

# --- Parameter ---
CONF_THRESHOLD  = 0.70
BATCH_SIZE      = 8
MAX_NEW_TOKENS  = 128
NUM_BEAMS       = 4

QWEN_MODEL_ID          = 'Qwen/Qwen3-4B'
QWEN_FALLBACK_MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device            : {device}')
print(f'TrOCR-Checkpoint  : {TROCR_CKPT}')
print(f'Konfidenz-Schwelle: {CONF_THRESHOLD}')
print(f'Qwen-Modell       : {QWEN_MODEL_ID}')
print(f'Debug-Logs        : {DEBUG_DIR}')

Device            : cuda
TrOCR-Checkpoint  : /home/justin/Ginger_Gradient/14/project/Capstone-Project/checkpoints/best_cook_large_v4
Konfidenz-Schwelle: 0.7
Qwen-Modell       : Qwen/Qwen3-4B
Debug-Logs        : /home/justin/Ginger_Gradient/14/project/Capstone-Project/data/debug_logs


In [2]:
with open(MANIFEST_PATH, encoding='utf-8') as f:
    global_manifest = json.load(f)

def sort_key(p):
    m = re.match(r'B(\d+)_P(\d+)', p['page_id'])
    return (int(m.group(1)), int(m.group(2))) if m else (99, 99)

pages = sorted(global_manifest['pages'], key=sort_key)

conf_done = sum(1 for p in pages if (CONF_DIR / f"{p['page_id']}.txt").exists())
corr_done = sum(1 for p in pages if (CORRECTED_DIR / f"{p['page_id']}.txt").exists())
print(f'Seiten gesamt              : {len(pages)}')
print(f'Konfidenz-Dateien vorhanden: {conf_done}')
print(f'Korrektur-Dateien vorhanden: {corr_done}')

Seiten gesamt              : 923
Konfidenz-Dateien vorhanden: 923
Korrektur-Dateien vorhanden: 0


---
## Stufe 1 – Konfidenz-Extraktion (TrOCR)

TrOCR wird mit `output_scores=True` ausgeführt.  
`compute_transition_scores` liefert Log-Wahrscheinlichkeiten pro Token.  
Wort-Konfidenz = geometrisches Mittel der Token-Log-Probs des Wortes: `exp(mean(log_probs))`.

In [3]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

gc.collect()
torch.cuda.empty_cache()

processor   = TrOCRProcessor.from_pretrained(TROCR_CKPT)
trocr_model = VisionEncoderDecoderModel.from_pretrained(TROCR_CKPT).to(device)
trocr_model.eval()

SEP_ID = processor.tokenizer.sep_token_id  # EOS-Token für TrOCR-Decoder
PAD_ID = processor.tokenizer.pad_token_id

print(f'TrOCR geladen : {TROCR_CKPT.name}')
print(f'Parameter     : {sum(p.numel() for p in trocr_model.parameters()):,}')
if torch.cuda.is_available():
    used_gb = torch.cuda.memory_allocated(device) / 1024**3
    total_gb = torch.cuda.get_device_properties(device).total_memory / 1024**3
    print(f'VRAM          : {used_gb:.1f} / {total_gb:.1f} GB')

Loading weights:   0%|          | 0/636 [00:00<?, ?it/s]

TrOCR geladen : best_cook_large_v4
Parameter     : 558,226,432
VRAM          : 2.1 / 11.6 GB


In [4]:
def ids_to_word_confidences(tok_ids: list, log_probs: list) -> list:
    """
    Mappt Token-IDs + Log-Probs auf Wörter.
    Gibt Liste von (wort: str, konfidenz: float, ist_unsicher: bool) zurück.
    
    RoBERTa-Tokenizer: Wort-Anfang → Token beginnt mit 'Ġ' (U+0120).
    Geometrisches Mittel: conf = exp(mean(log_probs_der_tokens)).
    """
    if not tok_ids:
        return []

    token_strs = processor.tokenizer.convert_ids_to_tokens(tok_ids)
    special    = {processor.tokenizer.eos_token, processor.tokenizer.bos_token,
                  processor.tokenizer.pad_token, '<s>', '</s>', '<pad>', None}

    words   = []
    cur_tok = []
    cur_lp  = []

    def flush():
        if not cur_tok:
            return
        word = ''.join(cur_tok).lstrip('Ġ▁')
        if word.strip():
            conf = float(np.exp(np.mean(cur_lp)))
            conf = max(0.0, min(1.0, conf))
            words.append((word, conf, conf < CONF_THRESHOLD))

    for tok, lp in zip(token_strs, log_probs):
        if tok in special:
            continue
        # Ġ / ▁ = neues Wort beginnt
        if (tok.startswith('Ġ') or tok.startswith('▁')) and cur_tok:
            flush()
            cur_tok, cur_lp = [tok], [lp]
        else:
            cur_tok.append(tok)
            cur_lp.append(lp)

    flush()
    return words


def process_page_confidences(line_paths: list) -> list:
    """
    Führt TrOCR mit output_scores=True (beam={NUM_BEAMS}) aus.
    Gibt Liste von (text: str, word_confs: list) pro Zeile zurück.
    """
    results = []

    for i in range(0, len(line_paths), BATCH_SIZE):
        batch = line_paths[i:i + BATCH_SIZE]
        imgs  = [Image.open(p).convert('RGB') for p in batch]
        pv    = processor(images=imgs, return_tensors='pt').pixel_values.to(device)

        with torch.no_grad():
            outputs = trocr_model.generate(
                pv,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=NUM_BEAMS,
                output_scores=True,
                return_dict_in_generate=True,
            )

        # beam_indices nötig für Beam-Search
        transition_scores = trocr_model.compute_transition_scores(
            outputs.sequences, outputs.scores, outputs.beam_indices,
            normalize_logits=True,
        )  # shape: (batch, max_new_tokens), log probs

        texts = processor.batch_decode(outputs.sequences, skip_special_tokens=True)

        for j in range(len(batch)):
            seq  = outputs.sequences[j]   # (1 + max_new_tokens,): [CLS, tok1, tok2, ...]
            t_sc = transition_scores[j]   # (max_new_tokens,)

            tok_ids, log_prs = [], []
            for k in range(t_sc.shape[0]):
                if k + 1 >= seq.shape[0]:
                    break
                tok_id = seq[k + 1].item()
                if tok_id in (SEP_ID, PAD_ID):
                    break
                tok_ids.append(tok_id)
                log_prs.append(t_sc[k].item())

            results.append((texts[j], ids_to_word_confidences(tok_ids, log_prs)))

        torch.cuda.empty_cache()

    return results


def save_confidence_file(page_id: str, line_results: list, out_path: Path) -> int:
    """
    Speichert Konfidenz-Datei.
    Format:
        === Buch X, Seite NNN ===
        SCHWELLE: 0.70
        UNSICHERE_WÖRTER: N von M

        [L01] wort1(0.98) <<wort2>>(0.42) wort3(0.91)
        ...
    Gibt Anzahl unsicherer Wörter zurück.
    """
    m = re.match(r'B(\d+)_P(\d+)', page_id)
    header = (f'=== Buch {m.group(1)}, Seite {m.group(2)} ===' if m
              else f'=== {page_id} ===')

    total_low   = sum(sum(1 for _, _, low in wc if low) for _, wc in line_results)
    total_words = sum(len(wc)                           for _, wc in line_results)

    lines_out = [
        header,
        f'SCHWELLE: {CONF_THRESHOLD}',
        f'UNSICHERE_WÖRTER: {total_low} von {total_words}',
        '',
    ]

    for idx, (text, word_confs) in enumerate(line_results, 1):
        if not word_confs:
            lines_out.append(f'[L{idx:02d}] {text}')
            continue
        parts = []
        for word, conf, is_low in word_confs:
            if is_low:
                parts.append(f'<<{word}>>({conf:.2f})')
            else:
                parts.append(f'{word}({conf:.2f})')
        lines_out.append(f'[L{idx:02d}] ' + ' '.join(parts))

    out_path.write_text('\n'.join(lines_out), encoding='utf-8')
    return total_low


print('Konfidenz-Funktionen definiert.')

Konfidenz-Funktionen definiert.


In [5]:
for page_entry in tqdm(pages, desc='Konfidenz-Extraktion'):
    page_id   = page_entry['page_id']
    conf_path = CONF_DIR / f'{page_id}.txt'

    if conf_path.exists():
        continue

    # Zeilen-Crops aus Seitenmanifest laden
    pm_path = REPO_ROOT / 'data' / 'all_manifests' / f'{page_id}.json'
    with open(pm_path, encoding='utf-8') as f:
        pm = json.load(f)

    line_paths = [
        REPO_ROOT / rec['line_image']
        for rec in pm['lines']
        if (REPO_ROOT / rec['line_image']).exists()
    ]

    if not line_paths:
        conf_path.write_text(
            f'=== {page_id} ===\nSCHWELLE: {CONF_THRESHOLD}\nUNSICHERE_WÖRTER: 0 von 0\n',
            encoding='utf-8',
        )
        continue

    line_results = process_page_confidences(line_paths)
    n_low = save_confidence_file(page_id, line_results, conf_path)

    tqdm.write(f'{page_id}: {len(line_results)} Zeilen, {n_low} unsichere Wörter')

print('\nKonfidenz-Extraktion abgeschlossen.')

Konfidenz-Extraktion: 100%|██████████| 923/923 [00:00<00:00, 109616.97it/s]


Konfidenz-Extraktion abgeschlossen.


In [6]:
# Statistik: Verteilung unsicherer Wörter pro Buch
from collections import defaultdict

book_stats = defaultdict(lambda: {'pages': 0, 'low': 0, 'total': 0})

for page_entry in pages:
    page_id   = page_entry['page_id']
    conf_path = CONF_DIR / f'{page_id}.txt'
    if not conf_path.exists():
        continue
    book = re.match(r'(B\d+)', page_id).group(1)
    for line in conf_path.read_text(encoding='utf-8').splitlines():
        if line.startswith('UNSICHERE_WÖRTER:'):
            parts = line.split(':')[1].strip().split()
            try:
                n_low, n_tot = int(parts[0]), int(parts[2])
                book_stats[book]['low']   += n_low
                book_stats[book]['total'] += n_tot
                book_stats[book]['pages'] += 1
            except (IndexError, ValueError):
                pass
            break

print(f"{'Buch':<6} {'Seiten':>7}  {'Unsichere':>10}  {'Gesamt':>8}  {'%-Anteil':>9}")
print('-' * 48)
for book in sorted(book_stats):
    s = book_stats[book]
    pct = 100 * s['low'] / s['total'] if s['total'] else 0
    print(f"{book:<6} {s['pages']:>7}  {s['low']:>10}  {s['total']:>8}  {pct:>8.1f}%")

Buch    Seiten   Unsichere    Gesamt   %-Anteil
------------------------------------------------
B1         132        2976     23593      12.6%
B2         164        5337     38667      13.8%
B3         151        5579     37629      14.8%
B4         165        7325     49039      14.9%
B5         155        8368     50982      16.4%
B6         156        8487     47630      17.8%


---
## Stufe 2 – Qwen Postkorrektur (2-Stufen-Pipeline)

**Zuerst TrOCR aus dem VRAM entladen** (Zelle unten), dann Qwen laden.

### Ablauf pro Seite:

**Step 1 – Analyse** (`temperature=0.3`, Thinking aktiviert)  
Qwen erhält den annotierten Text und liefert eine Vorschlagsliste:
- Für jedes `<<wort>>` (niedrige Konfidenz) **und** für klar unsinnige nicht-markierte Wörter  
- Format: `<<original>> → ersatz (reason: kurze Begründung)`  
- Ersatzwort muss dem Original optisch ähneln

**Step 2 – Korrektur** (`temperature=0.1`, greedy, Thinking deaktiviert)  
Qwen erhält Original-Text + Vorschlagsliste und gibt **nur** den bereinigten Text aus.

Debug-Logs (Step-1-Vorschläge) werden als JSON in `data/debug_logs/` gespeichert.

In [7]:
# TrOCR entladen bevor Qwen geladen wird
if 'trocr_model' in dir():
    del trocr_model
    gc.collect()
    torch.cuda.empty_cache()
    print('TrOCR-Modell aus VRAM entladen.')
else:
    gc.collect()
    torch.cuda.empty_cache()
    print('TrOCR nicht geladen – überspringe.')
if torch.cuda.is_available():
    used_gb = torch.cuda.memory_allocated(device) / 1024**3
    print(f'VRAM jetzt: {used_gb:.2f} GB belegt')

TrOCR-Modell aus VRAM entladen.
VRAM jetzt: 0.00 GB belegt


In [8]:
import importlib
import subprocess
import sys
from transformers import AutoModelForCausalLM, AutoTokenizer

# ftfy für Encoding-Artefakte (Mojibake-Bereinigung vor LLM-Aufruf)
if importlib.util.find_spec('ftfy') is None:
    print('Installiere ftfy...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy'])
import ftfy  # noqa: E402


def _load_fp16(model_id):
    return AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map='auto'
    )

def _load_4bit(model_id):
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    return AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb, device_map='auto'
    )

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID)

# Strategie:
# 1. Qwen3-4B float16 (~8-10 GB) – kein bitsandbytes nötig
# 2. Fallback: bitsandbytes installieren + 4-bit
# 3. Letzter Fallback: kleineres Modell float16

try:
    qwen_model = _load_fp16(QWEN_MODEL_ID)
    load_mode  = 'float16'
except torch.cuda.OutOfMemoryError:
    print('float16 zu groß – versuche 4-bit...')
    if importlib.util.find_spec('bitsandbytes') is None:
        print('Installiere bitsandbytes...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                               'bitsandbytes>=0.46.1'])
    try:
        qwen_model = _load_4bit(QWEN_MODEL_ID)
        load_mode  = '4-bit NF4'
    except Exception as e:
        print(f'4-bit fehlgeschlagen ({e}) – Fallback auf {QWEN_FALLBACK_MODEL_ID}')
        QWEN_MODEL_ID  = QWEN_FALLBACK_MODEL_ID
        qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID)
        qwen_model     = _load_fp16(QWEN_MODEL_ID)
        load_mode      = 'float16 (Fallback)'

qwen_model.eval()
print(f'Qwen geladen ({load_mode}): {QWEN_MODEL_ID}')
if torch.cuda.is_available():
    used_gb  = torch.cuda.memory_allocated(device) / 1024**3
    total_gb = torch.cuda.get_device_properties(device).total_memory / 1024**3
    print(f'VRAM: {used_gb:.1f} / {total_gb:.1f} GB')

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen geladen (float16): Qwen/Qwen3-4B
VRAM: 7.5 / 11.6 GB


In [9]:
_IS_QWEN3 = 'Qwen3' in QWEN_MODEL_ID

# ---------------------------------------------------------------------------
# Forster/Cook Kontext – konstant für alle Seiten, hartkodiert ins System-Prompt
# ---------------------------------------------------------------------------
_CONTEXT = """\
You are an expert in historical manuscript transcription.
Context: Johann Reinhold Forster's journal of Captain Cook's second voyage \
(HMS Resolution, 1772–1774).
Language: late 18th-century English and German, with nautical, botanical, \
and zoological terminology.
Topics: navigation, natural history (botany, zoology), encounters with Pacific \
indigenous peoples, shipboard life, Forster's relationship with Cook and the Admiralty.\
"""

STEP1_SYSTEM = _CONTEXT + """

You will receive an OCR-transcribed manuscript page.
Words with low OCR confidence are marked <<word>>.

Your task: identify ALL words that need correction:
  1. Every <<word>> that does not fit the context
  2. Any clearly nonsensical or misspelled UNMARKED word \
(garbled sequences, digits replacing letters, truncated words)

For each word to correct, output exactly one line:
  <<original>> → replacement  (reason: brief justification)

If a marked word already fits the context well, output:
  <<original>> → [keep]

Rules:
- The replacement MUST visually resemble the original (similar letters, similar length)
- Do NOT invent a word that looks nothing like the original
- Output ONLY the suggestion list — no introduction, no trailing text
"""

STEP2_SYSTEM = _CONTEXT + """

You will receive:
1. The OCR text with <<word>> markers for uncertain words
2. A correction suggestion list from a first analysis pass

Your task: produce the final corrected text.
- Apply all suggestions (ignore [keep] entries — leave those words unchanged)
- Remove ALL <<>> markers (keep the word as-is if no suggestion covers it)
- Do NOT change anything that is not listed in the suggestions
- Preserve line breaks and the page header (=== Buch X, Seite NNN ===) exactly
- Output ONLY the corrected text — no markers, no explanations, nothing else
"""


# ---------------------------------------------------------------------------
# Hilfs-Funktionen
# ---------------------------------------------------------------------------

def clean_encoding(text: str) -> str:
    """Bereinigt Mojibake und Encoding-Artefakte via ftfy."""
    return ftfy.fix_text(text)


def conf_file_to_annotated(conf_path: Path) -> tuple:
    """
    Liest Konfidenz-Datei → (annotated_text: str, n_low: int).
    Konvertiert '[L01] wort(0.98) <<w2>>(0.42)' → 'wort <<w2>>'.
    """
    lines     = conf_path.read_text(encoding='utf-8').splitlines()
    out_lines = []
    n_low     = 0

    for line in lines:
        if line.startswith('SCHWELLE') or not line.strip():
            continue
        if line.startswith('UNSICHERE_WÖRTER:'):
            try:
                n_low = int(line.split(':')[1].strip().split()[0])
            except (IndexError, ValueError):
                pass
            continue
        if line.startswith('==='):
            out_lines.append(line)
            continue

        parts     = line.split('] ', 1)
        word_part = parts[1] if len(parts) == 2 else line
        word_part = re.sub(r'<<([^>]+)>>\([\d.]+\)', r'<<\1>>', word_part)
        word_part = re.sub(r'(\S+)\([\d.]+\)', r'\1', word_part)
        out_lines.append(word_part.strip())

    return '\n'.join(out_lines), n_low


def _run_model(messages: list, enable_thinking: bool,
               temperature: float, max_new_tokens: int) -> str:
    """Führt Qwen mit gegebenen Parametern aus; gibt Antworttext zurück."""
    # Qwen3: thinking via /no_think steuern (bewährt, kein apply_chat_template kwarg nötig)
    if _IS_QWEN3 and not enable_thinking:
        msgs = list(messages)
        msgs[-1] = {**msgs[-1], 'content': msgs[-1]['content'] + '\n/no_think'}
    else:
        msgs = messages

    prompt = qwen_tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True,
    )
    inputs = qwen_tokenizer(prompt, return_tensors='pt').to(qwen_model.device)

    do_sample   = temperature > 0.05
    gen_kwargs  = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        pad_token_id=qwen_tokenizer.eos_token_id,
    )
    if do_sample:
        gen_kwargs['temperature'] = temperature

    with torch.no_grad():
        output_ids = qwen_model.generate(**inputs, **gen_kwargs)

    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    result     = qwen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    result     = re.sub(r'<think>.*?</think>', '', result, flags=re.DOTALL).strip()
    return result


def validate_output(text: str) -> tuple:
    """
    Prüft ob <<>> Marker noch vorhanden sind.
    Gibt (is_clean: bool, cleaned_text: str) zurück.
    """
    has_markers = bool(re.search(r'<<[^>]+>>', text))
    cleaned     = re.sub(r'<<([^>]+)>>', r'\1', text)
    return (not has_markers), cleaned


def postprocess_with_qwen(annotated_text: str, page_id: str) -> tuple:
    """
    Zwei-Stufen-Korrektur für eine Seite.

    Step 1 – Analyse (temperature=0.3, thinking aktiviert):
      Qwen listet Korrekturen für <<markierte>> UND klar unsinnige nicht-markierte Wörter.

    Step 2 – Korrektur (temperature=0.1, greedy, thinking deaktiviert):
      Qwen wendet Vorschläge an und gibt nur den bereinigten Text zurück.

    Returns (corrected_text: str, suggestions: str, is_clean: bool).
    """
    text   = clean_encoding(annotated_text)
    n_words = len(text.split())

    # Step 1: Suggestions
    step1_msgs = [
        {'role': 'system', 'content': STEP1_SYSTEM},
        {'role': 'user',   'content': text},
    ]
    suggestions = _run_model(
        step1_msgs,
        enable_thinking=True,
        temperature=0.3,
        max_new_tokens=max(256, n_words * 2),
    )

    # Step 2: Korrektur anwenden
    step2_input = (
        f'Original text (with <<>> markers):\n{text}\n\n'
        f'Correction suggestions:\n{suggestions}'
    )
    step2_msgs = [
        {'role': 'system', 'content': STEP2_SYSTEM},
        {'role': 'user',   'content': step2_input},
    ]
    corrected = _run_model(
        step2_msgs,
        enable_thinking=False,
        temperature=0.1,
        max_new_tokens=max(512, n_words * 4),
    )

    is_clean, corrected = validate_output(corrected)
    if not is_clean:
        tqdm.write(f'  [WARN] {page_id}: Marker nach Step 2 noch vorhanden – bereinigt')

    return corrected, suggestions, is_clean


print('Qwen-Funktionen (2-Stufen-Pipeline) definiert.')

Qwen-Funktionen (2-Stufen-Pipeline) definiert.


In [10]:
n_corrected  = 0
n_copied     = 0
n_skipped    = 0
n_flagged    = 0   # Seiten mit Marker-Warnung → manuelle Prüfung empfohlen

for page_entry in tqdm(pages, desc='Qwen Postkorrektur (2-Stufen)'):
    page_id        = page_entry['page_id']
    conf_path      = CONF_DIR      / f'{page_id}.txt'
    corrected_path = CORRECTED_DIR / f'{page_id}.txt'
    orig_path      = TRANSCR_DIR   / f'{page_id}.txt'
    debug_path     = DEBUG_DIR     / f'{page_id}.json'

    if corrected_path.exists():
        n_skipped += 1
        continue
    if not conf_path.exists():
        continue

    annotated_text, n_low = conf_file_to_annotated(conf_path)

    if n_low == 0:
        # Keine unsicheren Wörter → Original direkt übernehmen
        if orig_path.exists():
            corrected_path.write_text(orig_path.read_text(encoding='utf-8'), encoding='utf-8')
        n_copied += 1
        continue

    corrected_text, suggestions, is_clean = postprocess_with_qwen(annotated_text, page_id)

    # Seitenkopf sicherstellen
    m = re.match(r'B(\d+)_P(\d+)', page_id)
    if m:
        header = f'=== Buch {m.group(1)}, Seite {m.group(2)} ==='
        if not corrected_text.startswith('==='):
            corrected_text = header + '\n' + corrected_text

    corrected_path.write_text(corrected_text, encoding='utf-8')

    # Debug-Log: Step-1-Vorschläge für Nachvollziehbarkeit speichern
    debug_data = {
        'page_id':           page_id,
        'n_low_conf':        n_low,
        'markers_clean':     is_clean,
        'step1_suggestions': suggestions,
    }
    debug_path.write_text(
        json.dumps(debug_data, ensure_ascii=False, indent=2), encoding='utf-8'
    )

    if not is_clean:
        n_flagged += 1

    n_corrected += 1
    tqdm.write(f'{page_id}: {n_low} unsichere Wörter')

print(f'\nQwen Postkorrektur abgeschlossen.')
print(f'  Qwen korrigiert : {n_corrected}')
print(f'  Direkt kopiert  : {n_copied}  (keine unsicheren Wörter)')
print(f'  Übersprungen    : {n_skipped}  (bereits vorhanden)')
if n_flagged:
    print(f'  Marker-Warnungen: {n_flagged}  → manuelle Prüfung empfohlen (siehe data/debug_logs/)')

Qwen Postkorrektur (2-Stufen):   0%|          | 0/923 [00:00<?, ?it/s]

Qwen Postkorrektur (2-Stufen):   0%|          | 1/923 [00:11<3:00:15, 11.73s/it]

B1_P012: 7 unsichere Wörter


Qwen Postkorrektur (2-Stufen):   0%|          | 2/923 [00:34<4:35:33, 17.95s/it]

B1_P014: 28 unsichere Wörter


...

B1_P073: 33 unsichere Wörter


Qwen Postkorrektur (2-Stufen):   4%|▍         | 36/923 [12:27<6:03:12, 24.57s/it]

B1_P074: 23 unsichere Wörter


Qwen Postkorrektur (2-Stufen):   4%|▍         | 36/923 [12:38<5:11:27, 21.07s/it]


KeyboardInterrupt: 

---
## Stufe 3 – corrected_raw_document.txt

In [ ]:
total_lines = 0
missing     = []

with open(CORR_DOC_PATH, 'w', encoding='utf-8') as out:
    for page_entry in pages:
        page_id  = page_entry['page_id']
        txt_path = CORRECTED_DIR / f'{page_id}.txt'

        if not txt_path.exists():
            missing.append(page_id)
            continue

        content = txt_path.read_text(encoding='utf-8').strip()
        out.write(content + '\n\n')
        total_lines += len([l for l in content.splitlines() if not l.startswith('===')])

print(f'corrected_raw_document.txt : {CORR_DOC_PATH}')
print(f'Seiten eingebunden         : {len(pages) - len(missing)} / {len(pages)}')
print(f'Zeilen gesamt              : {total_lines}')
if missing:
    print(f'Fehlende Seiten            : {len(missing)}')

In [ ]:
# Vorschau: originale vs. korrigierte Transkription einer Beispielseite
import matplotlib.pyplot as plt

# Erste Seite mit tatsächlichen Korrekturen finden
sample_id = None
for page_entry in pages:
    pid = page_entry['page_id']
    cp  = CONF_DIR / f'{pid}.txt'
    if cp.exists():
        for line in cp.read_text(encoding='utf-8').splitlines():
            if line.startswith('UNSICHERE_WÖRTER:'):
                try:
                    if int(line.split(':')[1].strip().split()[0]) > 0:
                        sample_id = pid
                except:
                    pass
                break
    if sample_id:
        break

if sample_id:
    orig_txt = (TRANSCR_DIR / f'{sample_id}.txt').read_text(encoding='utf-8') if (TRANSCR_DIR / f'{sample_id}.txt').exists() else '(nicht vorhanden)'
    corr_txt = (CORRECTED_DIR / f'{sample_id}.txt').read_text(encoding='utf-8') if (CORRECTED_DIR / f'{sample_id}.txt').exists() else '(nicht vorhanden)'
    conf_txt = (CONF_DIR / f'{sample_id}.txt').read_text(encoding='utf-8')

    print(f'Beispielseite: {sample_id}')
    print('=' * 60)
    print('--- KONFIDENZ-DATEI (Auszug) ---')
    print('\n'.join(conf_txt.splitlines()[:12]))
    print()
    print('--- ORIGINAL-TRANSKRIPTION (Auszug) ---')
    print('\n'.join(orig_txt.splitlines()[:8]))
    print()
    print('--- KORRIGIERTE TRANSKRIPTION (Auszug) ---')
    print('\n'.join(corr_txt.splitlines()[:8]))
else:
    print('Noch keine Korrekturen vorhanden (Stufen 1 und 2 zuerst ausführen).')